# Task: Data analysis

Let us start by restating our working hypothesis:

**Hypothesis 1:** Important events will trigger a peak in twitter activity. 

**Hypothesis 2:** The biggest events will have an impact globaly. 

**Hypothesis 3:** Some peak twitter activity can be matched to events reported by mainstream media.

In the previous task we cleaned and pre-processed twitter daily count data for 2021 to extract peak event dates and number of tweets worldwide, in the USA and in the UK.

We will first compare twitter activity for those key event dates between contries and worldwide but before that we need to setup our coding environment by running the cell below.

In [254]:
%matplotlib inline
import pandas as pd
import matplotlib.pyplot as plt
#import seaborn as sns
from sklearn.linear_model import LinearRegression
import datetime as dt
import numpy as np
from scipy.stats import pearsonr
from sklearn.cluster import KMeans
#sns.set()

Now let us read the cleaned dataset from the previous task. We will call the dataframe **processed**.

In [256]:
processed = pd.read_excel('./input_data/processed_data.xlsx', sheet_name='processed',engine='openpyxl')


Let us look at the data again using the heat map option with pandas.

In [259]:
processed.style.background_gradient(cmap='Blues')

Looking qualitatively at the data we can already see that there are dates where there is high twitter activity in both the US and the UK and some also correspond to peak activity worldwide. But we also see that sometimes, high twitter activity is present only in one country and not the other.
How can we measure and validate such a relationship between the different activities? We can use a correlation analysis. 

## Correlation analysis

As introduced in <font color='blue'>**Topic correlation**</font>, correlation coefficients will help you identify which countries have a similar or a desimilar Twitter activity. Not only that but the strength of the coefficients will reflect the strength of this relationship and the sign of the coefficient will correspond to the direction of this relatonship. A positive correlation means that when Twitter activity increase for one country it will also increase in the other country. A negative correlation means the opposite. When Twitter activity increase for one country, it will decrease in the other one.

To run a correlation on the cleaned dataset we just need to apply the **corr** function from pandas to the dataset. There are different correlation methods. We will chose the pearson calculation.

In [ ]:
processed.corr(method='pearson')

What you see above is the correlation matrix between the different attributes. To tell if an attribute is correlated with another, you need to check the coefficient at the intersection of the row and column corresponding to those 2 attributes. For example, the correlation coefficient between worldwide and USA is 0.2. The diagonal of the matrix is always 1 because an attribute is always perfectly correlated to itself and the matrix is symetric because the correlation between an attribute A and an attribute B is the same as between B and A. 

Concerning the interpretation of the coefficients, close to 0 means no correlation, close to 1 is strong positive correlation and close to -1 is strong negative correlation. 

<font color='blue'>**All following interpretation will probably be different when we recieve the real data**</font>

In our case, there is a weak positive correlation between worldwide twitter activity and US (coefficient of 0.2) and a weak negative correlation between the UK and the USA (coefficients of -0.2). In the other hand we see a strong correlation between UK Twitter activity and worldwide (coefficient of 0.7).

how confident are we in our results? Could these relationships we found be due to chance? We have discussed the importance of validating your results in **Topic model validation**. One way to validate correlation relationships is to caclculate the p-value. 

## Model validation



In [247]:
def pearsonr_pval(x,y):
        return pearsonr(x,y)[1]

how confident are we in our results? Could these relationships we find be due to chance? We have discussed the importance of validating your results in **Topic model validation**. One way to validate correlation relationships is to caclculate the p-value. 

In [ ]:
processed.corr(method=pearsonr_pval)

In [ ]:
processed

In [ ]:
data_clustering=pd.melt(processed, id_vars=['day'], value_vars=['worldwide_events','USA_events','UK_events'])
data_clustering['day_ordinal']=data_clustering['day'].map(dt.datetime.toordinal)

km = KMeans(n_clusters=3)
km.fit(data_clustering[['day_ordinal','value']])

data_clustering['labels']=km.labels_
data_clustering

data_clustering.plot.scatter('day', 'value', c='labels', colormap='gist_rainbow')

In [250]:
mainstream_media = pd.read_excel('./input_data/mainstream_media.xlsx', sheet_name='data',engine='openpyxl')

In [ ]:
mainstream_media

In [252]:
processed.set_index('day',inplace=True)

In [253]:
processed[processed > 0.]=1.

In [ ]:
processed

In [235]:
mainstream_media.set_index('day',inplace=True)

In [236]:
media_2021=processed.join(mainstream_media,how="outer",sort=True)

In [237]:
media_2021.fillna(0,inplace=True)

In [ ]:
media_2021.style.background_gradient(cmap='Blues')

In [239]:
media_2021['media_num'] = media_2021['events'].apply(lambda x: x if x == 0 else 1)

#convert_objects(convert_numeric=True).fillna(1)

In [ ]:
media_2021

In [ ]:
media_2021.corr(method='pearson')

In [ ]:
media_2021.corr(method=pearsonr_pval)